# Gold Layer — fact_sales
Build sales fact table joining silver sales with gold dimensions.

## Setup Connection

In [ ]:
import os
from dotenv import load_dotenv
from clickzetta.zettapark.session import Session
from clickzetta.zettapark import functions as F
from clickzetta.zettapark.window import Window

load_dotenv()
session = Session.builder.configs({
    "username":  os.environ["CLICKZETTA_USERNAME"],
    "password":  os.environ["CLICKZETTA_PASSWORD"],
    "service":   os.environ["CLICKZETTA_SERVICE"],
    "instance":  os.environ["CLICKZETTA_INSTANCE"],
    "workspace": os.environ["CLICKZETTA_WORKSPACE"],
    "schema":    os.environ["CLICKZETTA_SCHEMA"],
    "vcluster":  os.environ["CLICKZETTA_VCLUSTER"],
}).create()
SCHEMA = os.environ["CLICKZETTA_SCHEMA"]

## The Transformation Logic

In [ ]:
sd = session.table(f"{SCHEMA}.crm_sales")
pr = session.table(f"{SCHEMA}.dim_products")
cu = session.table(f"{SCHEMA}.dim_customers")

joined = (
    sd.join(pr, sd["product_number"] == pr["product_number"], "left")
      .join(cu, sd["customer_id"] == cu["customer_id"], "left")
)

df = joined.select(
    sd["order_number"].alias("order_number"),
    pr["product_key"].alias("product_key"),
    cu["customer_key"].alias("customer_key"),
    sd["order_date"].alias("order_date"),
    sd["ship_date"].alias("ship_date"),
    sd["due_date"].alias("due_date"),
    sd["sales_amount"].alias("sales_amount"),
    sd["quantity"].alias("quantity"),
    sd["price"].alias("price"),
)

## Sanity Check

In [ ]:
df.limit(10).show()

## Write Gold Table

In [ ]:
df.write.save_as_table(f"{SCHEMA}.fact_sales", mode="overwrite")
print("fact_sales OK")

## Verify

In [ ]:
session.table(f"{SCHEMA}.fact_sales").limit(5).show()